# nightsearch-SAST Demo Walkthrough

This notebook is a presentation-focused walkthrough of the `nightsearch-SAST` repository for spatial transcriptomics spot annotation/deconvolution. The project compares a cross-attention model against an NNLS baseline under a shared evaluation schema. We support both a synthetic pipeline (for controlled experiments) and a first real-data pipeline path (normalized DLPFC-style NPZ bundles).

**Problem framing:** given spot-level expression and a reference dictionary, estimate cell-type composition per spot and evaluate reconstruction/annotation quality consistently across methods.


## 1) What this repo does

### Two runnable paths
- **Synthetic pipeline**: generate synthetic dictionary-style spot/reference data and run quick training/evaluation.
- **Real-data pipeline (v1)**: consume a normalized NPZ bundle (or split NPZ files), align genes, split deterministically, and compare cross-attention vs NNLS.

### Shared flow

```text
load data
  -> build/align reference dictionary
  -> deterministic train/validation split
  -> run cross-attention and NNLS
  -> compare under shared metrics schema
```

The key engineering point is the **unified real-data contract + shared metrics schema**, which makes method comparisons consistent and reproducible.


In [ ]:
# Notebook setup: robust path handling + concise display helpers
from __future__ import annotations

import json
import os
import subprocess
import sys
import warnings
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    # Allows opening notebook from notebook subdir in Jupyter
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src").exists() and (p / "configs").exists():
            REPO_ROOT = p
            break

SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))


def run_cmd(cmd: list[str], cwd: Path | None = None) -> subprocess.CompletedProcess:
    env = dict(os.environ)
    env["PYTHONPATH"] = str(SRC_PATH)
    return subprocess.run(cmd, cwd=cwd or REPO_ROOT, env=env, text=True, capture_output=True, check=False)


def show_yaml(path: Path) -> None:
    cfg = yaml.safe_load(path.read_text())
    display(Markdown(f"**`{path.relative_to(REPO_ROOT)}`**"))
    print(yaml.safe_dump(cfg, sort_keys=False))


def load_json(path: Path) -> dict:
    return json.loads(path.read_text())


print(f"Repo root: {REPO_ROOT}")
print(f"Pythonpath src: {SRC_PATH}")


## 2) Repo architecture tour

A clean, selective tour (not a full noisy tree):

In [ ]:
# Curated architecture view
key_paths = [
    "configs",
    "scripts",
    "src/nightsearch_sast",
    "tests",
    "reports",
    "README.md",
]

rows = []
for rel in key_paths:
    p = REPO_ROOT / rel
    kind = "dir" if p.is_dir() else "file"
    if p.is_dir():
        entries = sorted(x.name for x in p.iterdir() if not x.name.startswith('.'))
        preview = ", ".join(entries[:8]) + (" ..." if len(entries) > 8 else "")
    else:
        preview = "top-level project summary and usage"
    rows.append({"path": rel, "type": kind, "preview": preview})

pd.DataFrame(rows)


### Unified pipeline diagram

In [ ]:
# Lightweight visual: unified synthetic/real flow
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 2.6))
ax.axis("off")

steps = [
    "Input\n(synthetic or real bundle)",
    "Gene alignment +\nreference dictionary",
    "Deterministic\ntrain/val split",
    "Cross-attention\ntraining/eval",
    "NNLS baseline\neval",
    "Shared metrics\ncomparison",
]

for i, step in enumerate(steps):
    x = i * 1.9
    ax.text(x, 0, step, ha="center", va="center", bbox=dict(boxstyle="round,pad=0.4", facecolor="#eef", edgecolor="#446"))
    if i < len(steps)-1:
        ax.annotate("", xy=(x+0.95,0), xytext=(x+0.35,0), arrowprops=dict(arrowstyle="->", lw=1.5))

ax.set_xlim(-1, (len(steps)-1)*1.9 + 1)
ax.set_ylim(-1, 1)
plt.show()


## 3) What we have learned so far

- The **Python-first normalized real-data bundle** is the right abstraction for reproducible training/evaluation.
- **Gene matching** between spot expression and reference data is a critical contract; without it, comparisons are invalid.
- **Deterministic train/validation splits** are essential for fair NNLS vs cross-attention comparisons.
- A **shared metrics interface** enables direct side-by-side method comparison, even when supervision is absent (metrics can be `null`).
- The first public **DLPFC target** gives a concrete real-data path while avoiding runtime R dependencies during training/evaluation.
- Current scope remains intentionally narrow: one clean real-data family path, smoke-sized demos, and lightweight reproducibility over broad benchmark claims.


## 4) Interactive demo controls

These controls are lightweight and intended for live walkthrough. If `ipywidgets` is unavailable, use the fallback parameters in the next cell.

In [ ]:
# Interactive controls (optional dependency)
controls = {
    "mode": "synthetic",
    "epochs": 1,
    "validation_fraction": 0.2,
    "seed": 7,
    "run_methods": "both",
}

try:
    import ipywidgets as widgets
    from IPython.display import display

    mode = widgets.Dropdown(options=["synthetic", "real"], value="synthetic", description="Mode:")
    epochs = widgets.IntSlider(value=1, min=1, max=5, step=1, description="Epochs:")
    val_frac = widgets.FloatSlider(value=0.2, min=0.1, max=0.5, step=0.05, description="Val frac:")
    seed = widgets.IntSlider(value=7, min=1, max=99, step=1, description="Seed:")
    methods = widgets.ToggleButtons(options=["both", "nnls_only"], value="both", description="Methods:")

    def _sync(_=None):
        controls.update({
            "mode": mode.value,
            "epochs": int(epochs.value),
            "validation_fraction": float(val_frac.value),
            "seed": int(seed.value),
            "run_methods": methods.value,
        })

    for w in [mode, epochs, val_frac, seed, methods]:
        w.observe(_sync, names="value")
    _sync()

    display(widgets.VBox([mode, epochs, val_frac, seed, methods]))
    display(Markdown("Use the controls, then run the next demo cell."))
except Exception as e:
    print("ipywidgets unavailable; using fallback controls dict.")
    print(f"Reason: {e}")

controls


## 5) Small model training demo (fast, live-friendly)

This cell:
1. shows the config(s)
2. runs tiny commands via existing repo entrypoints
3. prints compact metrics tables
4. falls back gracefully if real bundle is absent


In [ ]:
ARTIFACTS = REPO_ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

# Ensure tiny real demo assets exist if real mode is requested or if we want both summaries
real_bundle = REPO_ROOT / "data/processed/real/dlpfc_151673_bundle.npz"
if not real_bundle.exists():
    _create = run_cmd([sys.executable, "scripts/create_real_example_data.py", "--output-dir", "data/processed/real", "--seed", str(controls["seed"])])
    if _create.returncode != 0:
        print("Could not create real demo data; continuing with synthetic-only outputs.")

# Build tiny synthetic config dynamically from smoke baseline
smoke_cfg = yaml.safe_load((REPO_ROOT / "configs/smoke.yaml").read_text())
smoke_cfg["seed"] = controls["seed"]
smoke_cfg["train"]["epochs"] = controls["epochs"]

notebook_synth_cfg = REPO_ROOT / "configs/notebook_smoke_demo.yaml"
notebook_synth_cfg.write_text(yaml.safe_dump(smoke_cfg, sort_keys=False), encoding="utf-8")

print("Synthetic config used:")
show_yaml(notebook_synth_cfg)

syn_out = ARTIFACTS / "notebook_synthetic_metrics.json"
syn_cmd = [sys.executable, "scripts/run_synthetic_baseline.py", "--config", str(notebook_synth_cfg), "--output", str(syn_out)]
syn_proc = run_cmd(syn_cmd)
print("Synthetic command return code:", syn_proc.returncode)
if syn_proc.returncode != 0:
    print(syn_proc.stderr[-1500:])

results = {}
if syn_out.exists():
    payload = load_json(syn_out)
    results["synthetic"] = payload

# Optionally run real pipeline path
run_real = controls["mode"] == "real" or controls["run_methods"] == "both"
real_cfg_path = REPO_ROOT / "configs/real_example.yaml"
if run_real and real_bundle.exists():
    real_cfg = yaml.safe_load(real_cfg_path.read_text())
    real_cfg["seed"] = controls["seed"]
    real_cfg["train"]["epochs"] = controls["epochs"]
    real_cfg["data"]["real"]["validation_fraction"] = controls["validation_fraction"]

    notebook_real_cfg = REPO_ROOT / "configs/notebook_real_demo.yaml"
    notebook_real_cfg.write_text(yaml.safe_dump(real_cfg, sort_keys=False), encoding="utf-8")

    print("\nReal config used:")
    show_yaml(notebook_real_cfg)

    real_out = ARTIFACTS / "notebook_real_metrics.json"
    real_cmd = [sys.executable, "scripts/run_real_pipeline.py", "--config", str(notebook_real_cfg), "--output", str(real_out)]
    real_proc = run_cmd(real_cmd)
    print("Real command return code:", real_proc.returncode)
    if real_proc.returncode != 0:
        print(real_proc.stderr[-1500:])
    if real_out.exists():
        results["real"] = load_json(real_out)
else:
    print("\nReal pipeline skipped (bundle missing or control set to synthetic-only).")

print("\nAvailable result payloads:", list(results.keys()))


## 6) Metrics comparison table (shared schema)

In [ ]:
def metric_row_from_real_payload(payload: dict, method_key: str) -> dict:
    m = payload.get(method_key, {})
    return {
        "run_type": "real",
        "method": m.get("method", method_key),
        "split": m.get("split", "validation"),
        "reconstruction_mse": m.get("reconstruction_mse"),
        "composition_mae": m.get("composition_mae"),
        "composition_kl": m.get("composition_kl"),
    }

rows = []

if "synthetic" in results:
    sm = results["synthetic"].get("metrics", {})
    rows.append({
        "run_type": "synthetic",
        "method": "cross_attention_train",
        "split": "validation",
        "reconstruction_mse": sm.get("val_loss_mean"),
        "composition_mae": None,
        "composition_kl": None,
    })
    rows.append({
        "run_type": "synthetic",
        "method": "nnls",
        "split": "validation",
        "reconstruction_mse": sm.get("nnls_val_loss_mean"),
        "composition_mae": None,
        "composition_kl": None,
    })

if "real" in results:
    rows.append(metric_row_from_real_payload(results["real"], "cross_attention"))
    rows.append(metric_row_from_real_payload(results["real"], "nnls"))

if rows:
    metrics_df = pd.DataFrame(rows)
    display(metrics_df.style.format(precision=6, na_rep="N/A"))
else:
    print("No metrics available yet. Run demo cell above first.")


## 7) Current limitations and next steps

### Current limitations
- Only the first real-data target family (DLPFC-style bundle path) is wired end-to-end.
- Real-data converters assume specific column/shape conventions that need validation for new datasets.
- Evaluation remains focused on shared reconstruction/composition metrics rather than broad biological benchmarking.

### Logical next milestones
1. Add additional public real-data targets behind the same bundle contract.
2. Expand evaluation with richer stratified analyses (region/sample-wise).
3. Add experiment tracking/report templates for reproducible demo-to-paper transitions.
4. Harden notebook demo config generation into a reusable script if repeatedly used in interviews/advisor updates.


---
### Demo notes for presenter
- If runtime is constrained, run synthetic only (`controls['mode']='synthetic'`).
- For real path, ensure `data/processed/real/dlpfc_151673_bundle.npz` exists (the demo cell attempts to create tiny example data automatically).
- All core runs use repository entrypoints instead of re-implementing training logic in notebook code.
